In [40]:
import numpy as np
import itertools
from collections import defaultdict

# Qiskit Optimization
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# Qiskit Algorithms & Primitives
from qiskit_algorithms.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver, SamplingVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

# Qiskit Circuits
from qiskit.circuit.library import TwoLocal
from qiskit.circuit import QuantumCircuit, ParameterVector

# ============================================================
# 1. PROBLEM SETUP
# ============================================================
dist = np.array([
    [0, 3.9, 6, 5],
    [4, 0, 3, 2],
    [6, 3, 0, 4],
    [5, 2, 4, 0],
], dtype=float)

scenic = {1: 5.0, 2: 9.0, 3: 6.0} 
pois = [1, 2, 3] # points of interest
slots = [1, 2]   # time slots
A, B = 12.0, 10.0 # penalty coefficients

# ============================================================
# 2. BUILD QUBO
# ============================================================
qp = QuadraticProgram("toy_hiking_route")
for i in pois:
    for p in slots:
        qp.binary_var(name=f"x_{i}_{p}")

linear, quadratic = defaultdict(float), defaultdict(float)
constant = 0.0

for i in pois:
    linear[f"x_{i}_1"] += dist[0][i] - scenic[i]
    linear[f"x_{i}_2"] += dist[i][0] - scenic[i]

for i in pois:
    for j in pois:
        quadratic[(f"x_{i}_1", f"x_{j}_2")] += dist[i][j]

for p in slots:
    constant += A
    for i in pois:
        linear[f"x_{i}_{p}"] += -A
    for i, j in itertools.combinations(pois, 2):
        quadratic[(f"x_{i}_{p}", f"x_{j}_{p}")] += 2 * A

for i in pois:
    quadratic[(f"x_{i}_1", f"x_{i}_2")] += B

qp.minimize(constant=constant, linear=dict(linear), quadratic=dict(quadratic))
op, offset = qp.to_ising()

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================
def decode_solution(xvec):
    var_names = [f"x_{i}_{p}" for i in pois for p in slots]
    sol = {name: int(round(val)) for name, val in zip(var_names, xvec)}
    first, second = None, None
    for i in pois:
        if sol[f"x_{i}_1"] == 1: first = i
        if sol[f"x_{i}_2"] == 1: second = i
    return [0, first, second, 0]

def create_hea_ansatz(qubits, layers):
    qc = QuantumCircuit(qubits)
    thetas = ParameterVector('theta', qubits * layers)

    for i in range(layers):
        # Rotation Layer
        for j in range(qubits):
            qc.ry(thetas[i * qubits + j], j)

        # Entanglement Layer (Ring topology)
        for j in range(qubits - 1):
            qc.cx(j, j + 1)
        if qubits > 1:
            qc.cx(qubits - 1, 0) # Close the ring

        if layers > 1:
            qc.barrier()
            
    return qc

# ============================================================
# 4. SOLVERS
# ============================================================
sampler = StatevectorSampler()

parallel_sampler = AerSampler()
parallel_sampler.options.backend_options.update({
    "method": "statevector",
    "max_parallel_threads": 0,           # Use all available CPU cores
    "statevector_parallel_threshold": 1,  # Force parallelism even for small toy qubits
    "device": "CPU"                       # Change to "GPU" if you have CUDA installed
})

# --- Exact ---
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)

# --- QAOA ---
qaoa = MinimumEigenOptimizer(QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=2)).solve(qp)

# --- Standard VQE (TwoLocal) ---
ansatz_twolocal = TwoLocal(num_qubits=op.num_qubits, rotation_blocks=["ry", "rz"], entanglement_blocks="cz", reps=3)
vqe = MinimumEigenOptimizer(SamplingVQE(sampler=sampler, ansatz=ansatz_twolocal, optimizer=COBYLA(maxiter=800))).solve(qp)

# --- Custom HEA VQE ---
custom_ansatz = create_hea_ansatz(op.num_qubits, layers=3)
custom_vqe_solver = SamplingVQE(sampler=sampler, ansatz=custom_ansatz, optimizer=COBYLA(maxiter=800))
custom_vqe_result = MinimumEigenOptimizer(custom_vqe_solver).solve(qp)

# ============================================================
# 5. OUTPUT
# ============================================================
print(f"\n{'Solver':<16} | {'Route':<15} | {'Cost':<10}")
print("-" * 47)
print(f"{'EXACT':<16} | {str(decode_solution(exact.x)):<15} | {exact.fval:<10.2f}")
print(f"{'QAOA':<16} | {str(decode_solution(qaoa.x)):<15} | {qaoa.fval:<10.2f}")
print(f"{'Standard VQE':<16} | {str(decode_solution(vqe.x)):<15} | {vqe.fval:<10.2f}")
print(f"{'Custom HEA VQE':<16} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}")

/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
/tmp/ipykernel_304550/3837801236.py:117: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz_twolocal = TwoLocal(num_qu


Solver           | Route           | Cost      
-----------------------------------------------
EXACT            | [0, 1, 2, 0]    | -1.10     
QAOA             | [0, 1, 2, 0]    | -1.10     
Standard VQE     | [0, 1, 2, 0]    | -1.10     
Custom HEA VQE   | [0, 1, 3, 0]    | -0.10     


In [ ]:
import numpy as np
import itertools
from collections import defaultdict

# Qiskit Optimization
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer

# Qiskit Algorithms & Primitives
from qiskit_algorithms.minimum_eigensolvers import QAOA, NumPyMinimumEigensolver, SamplingVQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

# --- NEW: Tensor Network Integration ---
from qiskit_aer.primitives import Sampler as AerSampler

# Qiskit Circuits
from qiskit.circuit.library import TwoLocal
from qiskit.circuit import QuantumCircuit, ParameterVector

# ============================================================
# 1. ACTUAL DATA SETUP (Extracted from Hackathon CSV)
# ============================================================
# 0: Benasque, 1: Cerler, 2: Ancils, 3: Portillón de Benás
dist = np.array([
    [0.00, 1.20, 0.45, 2.50],  # From Benasque
    [1.20, 0.00, 1.30, 3.00],  # From Cerler
    [0.45, 1.30, 0.00, 1.42],  # From Ancils (1.42h to Portillon in CSV)
    [2.50, 3.00, 1.42, 0.00],  # From Portillón
], dtype=float)

# Scenic values based on Elevation (Normalized to toy scale)
# Cerler (1530m), Ancils (1140m), Portillón (2444m)
scenic = {1: 6.51, 2: 5.02, 3: 10.0} 

pois = [1, 2, 3] # points of interest
slots = [1, 2]   # time slots
A, B = 15.0, 12.0 # Penalty coefficients (Slightly higher for real distance scale)

# ============================================================
# 2. BUILD QUBO
# ============================================================
qp = QuadraticProgram("benasque_hiking_route")
for i in pois:
    for p in slots:
        qp.binary_var(name=f"x_{i}_{p}")

linear, quadratic = defaultdict(float), defaultdict(float)
constant = 0.0

for i in pois:
    linear[f"x_{i}_1"] += dist[0][i] - scenic[i]
    linear[f"x_{i}_2"] += dist[i][0] - scenic[i]

for i in pois:
    for j in pois:
        quadratic[(f"x_{i}_1", f"x_{j}_2")] += dist[i][j]

for p in slots:
    constant += A
    for i in pois:
        linear[f"x_{i}_{p}"] += -A
    for i, j in itertools.combinations(pois, 2):
        quadratic[(f"x_{i}_{p}", f"x_{j}_{p}")] += 2 * A

for i in pois:
    quadratic[(f"x_{i}_1", f"x_{i}_2")] += B

qp.minimize(constant=constant, linear=dict(linear), quadratic=dict(quadratic))
op, offset = qp.to_ising()

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================
def decode_solution(xvec):
    var_names = [f"x_{i}_{p}" for i in pois for p in slots]
    sol = {name: int(round(val)) for name, val in zip(var_names, xvec)}
    first, second = None, None
    for i in pois:
        if sol[f"x_{i}_1"] == 1: first = i
        if sol[f"x_{i}_2"] == 1: second = i
    return [0, first, second, 0]

def create_hea_ansatz(qubits, layers):
    qc = QuantumCircuit(qubits)
    thetas = ParameterVector('theta', qubits * layers)
    for i in range(layers):
        for j in range(qubits):
            qc.ry(thetas[i * qubits + j], j)
        for j in range(qubits - 1):
            qc.cx(j, j + 1)
        if qubits > 1:
            qc.cx(qubits - 1, 0)
        if layers > 1:
            qc.barrier()
    return qc

# ============================================================
# 4. SOLVERS (Now with Tensor Network Integration)
# ============================================================
sampler = StatevectorSampler()

# --- TENSOR NETWORK Integration (MPS Method) ---
# This uses the Matrix Product State simulator for scalability
tn_sampler = AerSampler(backend_options={
    "method": "matrix_product_state",
    "matrix_product_state_max_bond_dimension": 16,
    "device": "CPU"
})

# --- Exact ---
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver()).solve(qp)

# --- QAOA ---
qaoa = MinimumEigenOptimizer(QAOA(sampler=sampler, optimizer=COBYLA(maxiter=300), reps=3)).solve(qp)

# --- Standard VQE ---
ansatz_twolocal = TwoLocal(num_qubits=op.num_qubits, rotation_blocks=["ry", "rz"], 
                           entanglement_blocks="cz", reps=3)
vqe = MinimumEigenOptimizer(SamplingVQE(sampler=sampler, ansatz=ansatz_twolocal, 
                                        optimizer=COBYLA(maxiter=800))).solve(qp)

# --- Custom HEA VQE ---
custom_ansatz = create_hea_ansatz(op.num_qubits, layers=4)
custom_vqe_solver = SamplingVQE(sampler=sampler, ansatz=custom_ansatz, optimizer=COBYLA(maxiter=800))
custom_vqe_result = MinimumEigenOptimizer(custom_vqe_solver).solve(qp)


# # --- Custom HEA VQE using TENSOR NETWORKS (MPS) ---
# custom_ansatz = create_hea_ansatz(op.num_qubits, layers=3)
# # We swap the sampler to tn_sampler here!
# custom_vqe_solver = SamplingVQE(sampler=tn_sampler, ansatz=custom_ansatz, optimizer=COBYLA(maxiter=800))
# custom_vqe_result = MinimumEigenOptimizer(custom_vqe_solver).solve(qp)

# ============================================================
# 5. OUTPUT
# ============================================================
print(f"\n{'Solver':<18} | {'Route':<15} | {'Cost':<10}")
print("-" * 50)
print(f"{'EXACT':<18} | {str(decode_solution(exact.x)):<15} | {exact.fval:<10.2f}")
print(f"{'QAOA':<18} | {str(decode_solution(qaoa.x)):<15} | {qaoa.fval:<10.2f}")
print(f"{'Standard VQE':<18} | {str(decode_solution(vqe.x)):<15} | {vqe.fval:<10.2f}")
print(f"{'Custom HEA VQE':<18} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}")
# print(f"{'Tensor Network VQE':<18} | {str(decode_solution(custom_vqe_result.x)):<15} | {custom_vqe_result.fval:<10.2f}")

/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/local/sugale/quantum_error_mitigation/benasquehack/lib/python3.12/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])
/tmp/ipykernel_322921/211166510.py:118: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz_twolocal = TwoLocal(num_qub


Solver             | Route           | Cost      
--------------------------------------------------
EXACT              | [0, 2, 3, 0]    | -10.65    
QAOA               | [0, 3, 2, 0]    | -10.65    
Standard VQE       | [0, 2, 3, 0]    | -10.65    
Custom HEA VQE     | [0, 3, 2, 0]    | -10.65    
